# Motor Exercise 1 — checking a motor-command routine

Plot the requested left and right PWM values against actual elapsed time. Use the plot to check the button-controlled start, three-second rotation, and explicit stop. A command trace is not a measurement of wheel speed.

The example data are deterministic and synthetic. They make the analysis runnable, but they are not middleware parameters or values your robot should reproduce. Replace the example with your own raw records and retain command, wheel, direction, trial, timing, and physical-condition information. Work through the synthetic example before substituting your measurements, then change only the example data or loading cell. Keep the raw columns unchanged and add derived quantities in new columns so every conclusion remains traceable to an observation.


In [ ]:
# Load numpy so its tools are available below.
import numpy as np
# Load pandas so its tools are available below.
import pandas as pd
# Load matplotlib.pyplot so its tools are available below.
import matplotlib.pyplot as plt
# Load seaborn so its tools are available below.
import seaborn as sns

# Choose a consistent seaborn visual style for every plot.
sns.set_theme(style="whitegrid", context="notebook")
# Change how pandas displays tables without changing their data.
pd.set_option("display.max_columns", 100)
# Create the reproducible random-number generator rng; the fixed seed makes the synthetic example repeatable.
rng = np.random.default_rng(2026)


## Example timestamped command record


In [ ]:
# Create an ordered NumPy set of numeric values and store it as elapsed_ms.
elapsed_ms = np.arange(0, 8_001, 100)
# Create a labelled pandas table and store it as command_log.
command_log = pd.DataFrame({"elapsed_ms": elapsed_ms})
# Calculate and store this derived value in the table field command_log['left_pwm'].
command_log["left_pwm"] = 0.0
# Calculate and store this derived value in the table field command_log['right_pwm'].
command_log["right_pwm"] = 0.0
# Calculate and store this derived value in the table field command_log['phase'].
command_log["phase"] = "waiting"
# Calculate and store this derived value in the table field command_log['button_pressed'].
command_log["button_pressed"] = command_log["elapsed_ms"] >= 2_000

# Calculate this expression and store the result as motor_on for later use.
motor_on = command_log["elapsed_ms"].between(2_000, 4_900)
# Calculate and store this derived value in the table field command_log.loc[motor_on, ['left_pwm', 'right_pwm']].
command_log.loc[motor_on, ["left_pwm", "right_pwm"]] = [-60, 60]
# Calculate and store this derived value in the table field command_log.loc[motor_on, 'phase'].
command_log.loc[motor_on, "phase"] = "motor_on"
# Calculate and store this derived value in the table field command_log.loc[command_log['elapsed_ms'] >= 5000, 'phase'].
command_log.loc[command_log["elapsed_ms"] >= 5_000, "phase"] = "stopped"

# command_log = pd.read_csv("motor_exercise01_commands.csv")
# Display the first few rows to check the table structure and values.
command_log.head()


## Check command bounds, phase duration, and safe stop


In [ ]:
# Calculate this expression and store the result as required for later use.
required = {
    "elapsed_ms", "left_pwm", "right_pwm", "phase", "button_pressed"
}
# Calculate this expression and store the result as missing for later use.
missing = required.difference(command_log.columns)
# Check this condition and run the indented commands only when it is true.
if missing:
    # Stop with a clear error because the data do not meet this requirement.
    raise ValueError(f"Missing columns: {sorted(missing)}")

# Calculate this expression and store the result as command_log for later use.
command_log = command_log.sort_values("elapsed_ms").reset_index(drop=True)
# Calculate and store this derived value in the table field command_log['interval_ms'].
command_log["interval_ms"] = command_log["elapsed_ms"].diff()
# Check this condition and run the indented commands only when it is true.
if (command_log["interval_ms"].dropna() <= 0).any():
    # Stop with a clear error because the data do not meet this requirement.
    raise ValueError("Timestamps must increase within the trial.")

# Group rows with matching labels, calculate summaries, and store the table as phase_duration.
phase_duration = (
    command_log.groupby("phase")["interval_ms"].sum().rename("duration_ms")
)
# Create a labelled one-dimensional pandas result called checks.
checks = pd.Series({
    "maximum_absolute_left_pwm": command_log["left_pwm"].abs().max(),
    "maximum_absolute_right_pwm": command_log["right_pwm"].abs().max(),
    "first_button_press_ms": command_log.loc[
        command_log["button_pressed"], "elapsed_ms"
    ].min(),
    "motor_on_duration_ms": phase_duration.get("motor_on", np.nan),
    "final_left_pwm": command_log.iloc[-1]["left_pwm"],
    "final_right_pwm": command_log.iloc[-1]["right_pwm"],
    "irregular_intervals": command_log["interval_ms"].nunique() > 1,
})
# Print a labelled result so it can be checked immediately.
print(checks)
# Display this value as the final output of the notebook cell.
phase_duration


## Plot requested PWM for both wheels


In [ ]:
# Reshape columns into tidy long-form rows and store them as plot_data.
plot_data = command_log.melt(
    id_vars=["elapsed_ms", "phase", "button_pressed"],
    value_vars=["left_pwm", "right_pwm"],
    var_name="wheel",
    value_name="PWM",
)
# Create plotting objects or plot data and store them as (fig, ax).
fig, ax = plt.subplots(figsize=(10, 4.8))
# Add this data or reference guide to the current figure.
sns.lineplot(
    data=plot_data,
    x="elapsed_ms",
    y="PWM",
    hue="wheel",
    drawstyle="steps-post",
    estimator=None,
    ax=ax,
)
# Add this data or reference guide to the current figure.
ax.axhline(0, color="black", linewidth=1)
# Set labels or presentation details that make the plot interpretable.
ax.set(
    title="Requested motor-command timeline",
    xlabel="Elapsed time (ms)",
    ylabel="PWM",
)
# Render and display the completed Matplotlib figure.
plt.show()


## Evidence questions

1. Does motor activation begin only after the recorded button press?
2. Does the measured motor-on interval support the intended three seconds?
3. Do opposite command signs produce the intended on-the-spot rotation?
4. Is the final command an explicit zero for both wheels?
5. Why does this command plot not establish that the wheels actually moved?
6. What timing and safety changes are needed before the physical run?
